## Importing Packages

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Regression Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

## DagsHub MLflow Setup

In [6]:
import dagshub

dagshub.init(
    repo_owner="",
    repo_name="MLflow-101",
    mlflow=True
)

Repository MLflow-101 doesn't exist, creating it under organization "sabarish2410229".

Response (500):
{"message":"GetOrgByName","status":"error","support_id":"8d89c7b10a448304e2e273b8ae448df3"}


RuntimeError: Failed to create the desired repository.

In [ ]:
mlflow.set_experiment(
    "Housing Price Prediction PBLM 1"
)

## Data Loading and Processing

In [ ]:
import pandas as pd

data = pd.read_csv(r"C:\Users\DELL\Downloads\HousingData.csv")

# Remove missing values
data = data.dropna()

# Features and Target
X = data.drop("MEDV", axis=1)
y = data["MEDV"]

print("Shape:", X.shape)
print("Target Column: MEDV")
print("Target Statistics:")
print(y.describe())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

## Build Models

In [ ]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost",
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            random_state=42
        ),
        X_train,
        y_train
    )
]

In [ ]:
reports = []
trained_models = []

for model_name, model, X_tr, y_tr in models:

    model.fit(X_tr, y_tr)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    report = {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

    reports.append(report)
    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)
    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

## Log All Experiments to DagsHub

In [ ]:
for i, (model_name, model, _, _) in enumerate(models):

    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        # Parameters
        mlflow.log_param("Model", model_name)
        mlflow.log_params(model.get_params())

        # Regression Metrics
        mlflow.log_metric("MAE", report["MAE"])
        mlflow.log_metric("MSE", report["MSE"])
        mlflow.log_metric("RMSE", report["RMSE"])
        mlflow.log_metric("R2", report["R2"])

        # Log Model
        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

print("Experiments logged successfully!")

## Best Model and Reg to DH

In [ ]:
best_index = np.argmax(
    [r["R2"] for r in reports]
)

best_model_name = models[best_index][0]
best_model = trained_models[best_index]
best_report = reports[best_index]

print("Best Model:", best_model_name)
print("R2 Score :", best_report["R2"])
print("MAE      :", best_report["MAE"])
print("MSE      :", best_report["MSE"])
print("RMSE     :", best_report["RMSE"])

In [ ]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:

    # Parameters
    mlflow.log_param(
        "Model",
        best_model_name
    )

    mlflow.log_param(
        "Selection_Metric",
        "R2 Score"
    )

    # Metrics
    mlflow.log_metric(
        "R2",
        best_report["R2"]
    )

    mlflow.log_metric(
        "MAE",
        best_report["MAE"]
    )

    mlflow.log_metric(
        "MSE",
        best_report["MSE"]
    )

    mlflow.log_metric(
        "RMSE",
        best_report["RMSE"]
    )

    # Log model
    if "XGBoost" in best_model_name:

        model_info = mlflow.xgboost.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Housing_Price_Best_Model"
        )

    else:

        model_info = mlflow.sklearn.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Housing_Price_Best_Model"
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID    :", run_id)
print("Model URI :", model_uri)
print("Model Name:", "Housing_Price_Best_Model")